# Skin Disease 95+ Upgrade (Multi-Seed + Bias Calibration + Pseudo-Label + Rich TTA)

This notebook extends the 0.9543 solution with:

1. `2 seeds x 2 models x K folds` ensemble averaging  
2. OOF per-class bias calibration (optimize macro-F1)  
3. High-confidence pseudo-label stage (`max_prob >= 0.97`)  
4. Rich TTA (8 views: flips + light scale transforms)


In [ ]:
import copy
import gc
import json
import os
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report

from torchvision import models
from torchvision.transforms import v2

print('torch:', torch.__version__)


In [ ]:
# =========================
# Config / paths
# =========================
LOCAL_BASE = Path('/Users/songling/Desktop/Skin Disease Classification')
PLATFORM_BASE = Path('dataset/public')

if PLATFORM_BASE.exists():
    MODE = 'platform'
    BASE_DIR = PLATFORM_BASE
    OUTPUT_DIR = Path('working')
elif LOCAL_BASE.exists():
    MODE = 'local'
    BASE_DIR = LOCAL_BASE
    OUTPUT_DIR = BASE_DIR
else:
    raise FileNotFoundError('Cannot find dataset path.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = OUTPUT_DIR / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
run_name = datetime.now().strftime('run_95plus_%Y%m%d_%H%M%S')
run_dir = RUNS_DIR / run_name
run_dir.mkdir(parents=True, exist_ok=False)

TRAIN_CSV = BASE_DIR / 'train.csv'
TEST_CSV = BASE_DIR / 'test.csv'
TRAIN_IMG_DIR = BASE_DIR / 'train'
TEST_IMG_DIR = BASE_DIR / 'test'

CLASS_NAMES = ['acne', 'eksim', 'herpes', 'panu', 'rosacea']
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}
idx2label = {i: c for c, i in label2idx.items()}
N_CLASSES = len(CLASS_NAMES)

SEEDS = [42, 2024]
N_SPLITS = 5
BATCH_SIZE = 16
IMG_SIZE = 224

EPOCHS = 14
HEAD_EPOCHS = 2
LR_HEAD = 1e-3
LR_FINE = 2e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
EARLY_STOP = 4
GRAD_CLIP_NORM = 1.0

# Pseudo-label stage
PSEUDO_THRESHOLD = 0.97
PSEUDO_EPOCHS = 4
PSEUDO_LR = 8e-5
PSEUDO_MIX_RATIO = 0.35  # final = (1-r)*stage1 + r*pseudo_refined

MODEL_CONFIGS = [
    {'name': 'swin_v2_t', 'weight': 0.55},
    {'name': 'efficientnet_v2_s', 'weight': 0.45},
]

NUM_WORKERS = 0 if MODE == 'local' else 2
PERSISTENT_WORKERS = NUM_WORKERS > 0

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

PIN_MEMORY = (device.type == 'cuda')
USE_AMP = (device.type == 'cuda')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print('Mode      :', MODE)
print('Base dir  :', BASE_DIR)
print('Output dir:', OUTPUT_DIR)
print('Run dir   :', run_dir)
print('Device    :', device)


In [ ]:
# =========================
# Data / transforms / utils
# =========================
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

assert set(train_df['disease'].unique()) == set(CLASS_NAMES), 'Class mismatch'
assert len(test_df) == 180, f'Test rows should be 180, got {len(test_df)}'

train_df['label'] = train_df['disease'].map(label2idx)

def make_class_weights(df):
    counts = df['label'].value_counts().sort_index().values
    w = len(df) / (N_CLASSES * counts)
    return torch.tensor(w, dtype=torch.float32)

class AddGaussianNoise(nn.Module):
    def __init__(self, std=0.02, p=0.25):
        super().__init__()
        self.std = std
        self.p = p

    def forward(self, x):
        if torch.rand(1).item() < self.p:
            x = torch.clamp(x + torch.randn_like(x) * self.std, 0.0, 1.0)
        return x

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_tfms = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.RandomHorizontalFlip(0.5),
    v2.RandomVerticalFlip(0.15),
    v2.RandomRotation(20),
    v2.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    v2.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.12, hue=0.03),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    AddGaussianNoise(std=0.02, p=0.25),
    v2.Normalize(MEAN, STD),
])

valid_tfms = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])

class SkinDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.image_dir / row['filename']).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        if self.is_test:
            return img, int(row['id'])
        return img, int(row['label'])


def build_loader(ds, shuffle):
    kwargs = dict(batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    if PERSISTENT_WORKERS:
        kwargs['persistent_workers'] = True
    return DataLoader(ds, **kwargs)


In [ ]:
# =========================
# Model / train / TTA / calibration
# =========================
def build_model(model_name):
    if model_name == 'swin_v2_t':
        m = models.swin_v2_t(weights=models.Swin_V2_T_Weights.IMAGENET1K_V1)
        in_features = m.head.in_features
        m.head = nn.Linear(in_features, N_CLASSES)
    elif model_name == 'efficientnet_v2_s':
        m = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        in_features = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_features, N_CLASSES)
    else:
        raise ValueError(model_name)
    return m.to(device)


def set_head_only(model, model_name, head_only):
    for p in model.parameters():
        p.requires_grad = not head_only
    if head_only:
        if model_name == 'swin_v2_t':
            for p in model.head.parameters():
                p.requires_grad = True
        else:
            for p in model.classifier[1].parameters():
                p.requires_grad = True


def pad_or_crop_to_size(x, target_h, target_w):
    _, _, h, w = x.shape
    if h > target_h:
        top = (h - target_h) // 2
        x = x[:, :, top:top + target_h, :]
    elif h < target_h:
        pad_total = target_h - h
        pad_top = pad_total // 2
        pad_bottom = pad_total - pad_top
        x = F.pad(x, (0, 0, pad_top, pad_bottom), mode='reflect')

    _, _, h, w = x.shape
    if w > target_w:
        left = (w - target_w) // 2
        x = x[:, :, :, left:left + target_w]
    elif w < target_w:
        pad_total = target_w - w
        pad_left = pad_total // 2
        pad_right = pad_total - pad_left
        x = F.pad(x, (pad_left, pad_right, 0, 0), mode='reflect')
    return x


def scale_view(x, scale):
    b, c, h, w = x.shape
    x2 = F.interpolate(x, scale_factor=scale, mode='bilinear', align_corners=False)
    x2 = pad_or_crop_to_size(x2, h, w)
    return x2


@torch.no_grad()
def tta_logits_8(model, x):
    # 8-view TTA: original + flips + light scales
    views = [
        x,
        torch.flip(x, dims=[3]),
        torch.flip(x, dims=[2]),
        torch.flip(x, dims=[2, 3]),
        scale_view(x, 0.92),
        scale_view(x, 1.08),
        torch.flip(scale_view(x, 0.92), dims=[3]),
        torch.flip(scale_view(x, 1.08), dims=[2]),
    ]
    out = 0
    for v in views:
        out = out + model(v)
    return out / len(views)


@torch.no_grad()
def predict_proba(model, loader, use_tta=True):
    model.eval()
    all_probs = []
    for batch in loader:
        images = batch[0].to(device, non_blocking=True)
        logits = tta_logits_8(model, images) if use_tta else model(images)
        if logits.shape[1] != N_CLASSES:
            raise RuntimeError(f'Invalid logits dim: {logits.shape[1]}')
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)


def evaluate_from_proba(y_true, proba):
    pred = np.argmax(proba, axis=1)
    macro = f1_score(y_true, pred, average='macro')
    pc = f1_score(y_true, pred, average=None, labels=list(range(N_CLASSES)))
    return float(macro), np.array(pc), pred


def optimize_class_bias(y_true, proba, class_names, rounds=3):
    # Coordinate search on logit biases to maximize macro-F1
    eps = 1e-8
    logp = np.log(np.clip(proba, eps, 1.0))
    bias = np.zeros(N_CLASSES, dtype=np.float32)

    def score(b):
        pred = np.argmax(logp + b[None, :], axis=1)
        return f1_score(y_true, pred, average='macro')

    best = score(bias)
    step_schedule = [0.35, 0.2, 0.1, 0.05]

    for _ in range(rounds):
        improved = False
        for step in step_schedule:
            for c in range(N_CLASSES):
                for delta in (-step, step):
                    cand = bias.copy()
                    cand[c] += delta
                    s = score(cand)
                    if s > best:
                        best = s
                        bias = cand
                        improved = True
        if not improved:
            break

    return bias, float(best)


def apply_bias_to_proba(proba, bias):
    eps = 1e-8
    logp = np.log(np.clip(proba, eps, 1.0)) + bias[None, :]
    logp = logp - logp.max(axis=1, keepdims=True)
    exp = np.exp(logp)
    return exp / exp.sum(axis=1, keepdims=True)


In [ ]:
# =========================
# Stage 1: multi-seed x multi-model x CV
# =========================
model_weight_sum = sum(m['weight'] for m in MODEL_CONFIGS)

oof_stage1 = np.zeros((len(train_df), N_CLASSES), dtype=np.float32)
test_stage1 = np.zeros((len(test_df), N_CLASSES), dtype=np.float32)

logs = []
ckpt_info = []

for seed in SEEDS:
    seed_everything(seed)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    for mcfg in MODEL_CONFIGS:
        model_name = mcfg['name']
        w_model = mcfg['weight'] / model_weight_sum

        model_oof = np.zeros((len(train_df), N_CLASSES), dtype=np.float32)
        model_test = np.zeros((len(test_df), N_CLASSES), dtype=np.float32)

        print(f"\n===== Seed {seed} | Model {model_name} =====")

        for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df['label']), start=1):
            print(f"\n--- Fold {fold}/{N_SPLITS} ---")

            tr_df = train_df.iloc[tr_idx].copy()
            va_df = train_df.iloc[va_idx].copy()

            tr_ds = SkinDataset(tr_df, TRAIN_IMG_DIR, transform=train_tfms, is_test=False)
            va_ds = SkinDataset(va_df, TRAIN_IMG_DIR, transform=valid_tfms, is_test=False)
            te_ds = SkinDataset(test_df, TEST_IMG_DIR, transform=valid_tfms, is_test=True)

            tr_loader = build_loader(tr_ds, shuffle=True)
            va_loader = build_loader(va_ds, shuffle=False)
            te_loader = build_loader(te_ds, shuffle=False)

            model = build_model(model_name)
            class_weights = make_class_weights(tr_df).to(device)
            criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

            set_head_only(model, model_name, head_only=True)
            opt_head = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)

            set_head_only(model, model_name, head_only=False)
            opt_full = AdamW(model.parameters(), lr=LR_FINE, weight_decay=WEIGHT_DECAY)
            sched = CosineAnnealingLR(opt_full, T_max=max(EPOCHS - HEAD_EPOCHS, 1), eta_min=1e-6)

            scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
            best_f1 = -1.0
            best_state = None
            bad = 0

            y_val_true = va_df['label'].values

            for epoch in range(1, EPOCHS + 1):
                model.train()
                losses = []

                if epoch <= HEAD_EPOCHS:
                    set_head_only(model, model_name, head_only=True)
                    optimizer = opt_head
                else:
                    set_head_only(model, model_name, head_only=False)
                    optimizer = opt_full

                for images, labels in tr_loader:
                    images = images.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)

                    optimizer.zero_grad(set_to_none=True)
                    with torch.amp.autocast('cuda', enabled=USE_AMP):
                        logits = model(images)
                        loss = criterion(logits, labels)

                    scaler.scale(loss).backward()
                    if USE_AMP:
                        scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                    losses.append(loss.item())

                if epoch > HEAD_EPOCHS:
                    sched.step()

                val_proba = predict_proba(model, va_loader, use_tta=False)
                val_f1, _, val_pred = evaluate_from_proba(y_val_true, val_proba)
                pred_dist = pd.Series(val_pred).value_counts(normalize=True).sort_index()
                dist = {idx2label[i]: round(float(pred_dist.get(i, 0.0)), 3) for i in range(N_CLASSES)}

                print(f"Epoch {epoch:02d}/{EPOCHS} train_loss={np.mean(losses):.4f} val_f1={val_f1:.4f} dist={dist}")
                if max(dist.values()) > 0.92:
                    print('Warning: possible class-collapse tendency.')

                if val_f1 > best_f1:
                    best_f1 = val_f1
                    best_state = copy.deepcopy(model.state_dict())
                    bad = 0
                else:
                    bad += 1

                if bad >= EARLY_STOP:
                    print(f'Early stop at epoch {epoch}')
                    break

            model.load_state_dict(best_state)

            va_proba = predict_proba(model, va_loader, use_tta=True)
            model_oof[va_idx] = va_proba

            te_proba = predict_proba(model, te_loader, use_tta=True)
            model_test += te_proba / N_SPLITS

            ckpt = run_dir / f'stage1_{model_name}_seed{seed}_fold{fold}.pt'
            torch.save(best_state, ckpt)
            ckpt_info.append({
                'model': model_name,
                'seed': seed,
                'fold': fold,
                'path': str(ckpt),
                'best_f1': float(best_f1),
            })
            logs.append({'stage': 'stage1', 'model': model_name, 'seed': seed, 'fold': fold, 'best_val_f1': float(best_f1)})

            del model, tr_ds, va_ds, te_ds, tr_loader, va_loader, te_loader
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        w_seed = 1.0 / len(SEEDS)
        oof_stage1 += model_oof * w_model * w_seed
        test_stage1 += model_test * w_model * w_seed

y_true = train_df['label'].values
oof_stage1_f1, oof_stage1_pc, oof_stage1_pred = evaluate_from_proba(y_true, oof_stage1)
print('\nStage1 OOF macro F1:', round(oof_stage1_f1, 6))


In [ ]:
# =========================
# Stage 1.5: OOF bias calibration
# =========================
bias_vec, bias_best = optimize_class_bias(y_true, oof_stage1, CLASS_NAMES, rounds=4)
print('Bias vector:', bias_vec)
print('OOF macro F1 after bias calibration:', round(bias_best, 6))

oof_bias = apply_bias_to_proba(oof_stage1, bias_vec)
oof_bias_f1, oof_bias_pc, oof_bias_pred = evaluate_from_proba(y_true, oof_bias)

print('\nClassification report (biased OOF):')
print(classification_report(
    y_true,
    oof_bias_pred,
    labels=list(range(N_CLASSES)),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

cm = confusion_matrix(y_true, oof_bias_pred, labels=list(range(N_CLASSES)))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Biased OOF Confusion Matrix')
plt.xlabel('Pred')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(run_dir / 'oof_confusion_matrix_biased.png', dpi=180)
plt.show()

test_stage1_biased = apply_bias_to_proba(test_stage1, bias_vec)


In [ ]:
# =========================
# Stage 2: high-confidence pseudo-label refinement
# =========================
pseudo_prob = test_stage1_biased
pseudo_conf = pseudo_prob.max(axis=1)
pseudo_label = pseudo_prob.argmax(axis=1)

pseudo_mask = pseudo_conf >= PSEUDO_THRESHOLD
pseudo_count = int(pseudo_mask.sum())
print(f'Pseudo selected: {pseudo_count}/{len(test_df)} at threshold {PSEUDO_THRESHOLD}')

if pseudo_count > 0:
    pseudo_df = test_df.loc[pseudo_mask, ['id', 'filename']].copy()
    pseudo_df['label'] = [int(x) for x in pseudo_label[pseudo_mask]]
    pseudo_df['disease'] = [idx2label[int(x)] for x in pseudo_df['label'].values]
else:
    pseudo_df = pd.DataFrame(columns=['id', 'filename', 'label', 'disease'])

pseudo_df.to_csv(run_dir / 'pseudo_selected.csv', index=False)

# Refine only if enough pseudo samples selected
test_stage2 = np.zeros_like(test_stage1)

if pseudo_count >= 8:
    for seed in SEEDS:
        seed_everything(seed)
        for mcfg in MODEL_CONFIGS:
            model_name = mcfg['name']
            w_model = mcfg['weight'] / model_weight_sum

            # Train once on full train + pseudo
            base_train = train_df[['filename', 'label', 'disease']].copy()
            pseudo_train = pseudo_df[['filename', 'label', 'disease']].copy()
            full_train = pd.concat([base_train, pseudo_train], axis=0).reset_index(drop=True)

            ds = SkinDataset(full_train, TRAIN_IMG_DIR if len(pseudo_train)==0 else BASE_DIR / 'train', transform=train_tfms, is_test=False)
            # pseudo images are from test dir, so dataset path must route per-row. We handle this with custom dataset below.
            class HybridPseudoDataset(Dataset):
                def __init__(self, real_df, pseudo_df, train_dir, test_dir, transform):
                    self.real_df = real_df.reset_index(drop=True)
                    self.pseudo_df = pseudo_df.reset_index(drop=True)
                    self.train_dir = Path(train_dir)
                    self.test_dir = Path(test_dir)
                    self.transform = transform
                    self.n_real = len(self.real_df)
                    self.n_pseudo = len(self.pseudo_df)

                def __len__(self):
                    return self.n_real + self.n_pseudo

                def __getitem__(self, idx):
                    if idx < self.n_real:
                        row = self.real_df.iloc[idx]
                        img = Image.open(self.train_dir / row['filename']).convert('RGB')
                        y = int(row['label'])
                    else:
                        row = self.pseudo_df.iloc[idx - self.n_real]
                        img = Image.open(self.test_dir / row['filename']).convert('RGB')
                        y = int(row['label'])
                    img = self.transform(img)
                    return img, y

            train_hybrid = HybridPseudoDataset(
                real_df=train_df[['filename', 'label']],
                pseudo_df=pseudo_df[['filename', 'label']],
                train_dir=TRAIN_IMG_DIR,
                test_dir=TEST_IMG_DIR,
                transform=train_tfms,
            )

            tr_loader = build_loader(train_hybrid, shuffle=True)
            te_ds = SkinDataset(test_df, TEST_IMG_DIR, transform=valid_tfms, is_test=True)
            te_loader = build_loader(te_ds, shuffle=False)

            model = build_model(model_name)
            cw = make_class_weights(train_df).to(device)
            criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=LABEL_SMOOTHING)
            opt = AdamW(model.parameters(), lr=PSEUDO_LR, weight_decay=WEIGHT_DECAY)
            scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

            for ep in range(1, PSEUDO_EPOCHS + 1):
                model.train()
                losses = []
                for images, labels in tr_loader:
                    images = images.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)

                    opt.zero_grad(set_to_none=True)
                    with torch.amp.autocast('cuda', enabled=USE_AMP):
                        logits = model(images)
                        loss = criterion(logits, labels)

                    scaler.scale(loss).backward()
                    if USE_AMP:
                        scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    scaler.step(opt)
                    scaler.update()
                    losses.append(loss.item())

                print(f'Pseudo refine | seed={seed} model={model_name} epoch={ep}/{PSEUDO_EPOCHS} loss={np.mean(losses):.4f}')

            te_prob = predict_proba(model, te_loader, use_tta=True)
            test_stage2 += te_prob * (w_model / len(SEEDS))

            del model, train_hybrid, tr_loader, te_ds, te_loader
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

else:
    print('Pseudo sample count too small; skip stage2 refinement.')

if pseudo_count >= 8:
    test_final = (1.0 - PSEUDO_MIX_RATIO) * test_stage1_biased + PSEUDO_MIX_RATIO * test_stage2
else:
    test_final = test_stage1_biased.copy()


In [ ]:
# =========================
# Final submission / artifacts
# =========================
final_label = np.argmax(test_final, axis=1)
submission = pd.DataFrame({
    'id': test_df['id'].values,
    'disease': [idx2label[int(i)] for i in final_label],
}).sort_values('id').reset_index(drop=True)

assert len(submission) == 180
assert submission['disease'].isin(CLASS_NAMES).all()

run_submission_path = run_dir / 'submission.csv'
platform_submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(run_submission_path, index=False)
submission.to_csv(platform_submission_path, index=False)

if MODE == 'local':
    local_submission_path = BASE_DIR / 'submission.csv'
    submission.to_csv(local_submission_path, index=False)

pd.DataFrame(logs).to_csv(run_dir / 'training_logs.csv', index=False)
pd.DataFrame(ckpt_info).to_csv(run_dir / 'stage1_ckpt_info.csv', index=False)

summary = {
    'run_name': run_name,
    'mode': MODE,
    'device': str(device),
    'seeds': SEEDS,
    'n_splits': N_SPLITS,
    'models': MODEL_CONFIGS,
    'stage1_oof_macro_f1': float(oof_stage1_f1),
    'biased_oof_macro_f1': float(oof_bias_f1),
    'bias_vector': [float(x) for x in bias_vec],
    'pseudo_threshold': float(PSEUDO_THRESHOLD),
    'pseudo_count': int(pseudo_count),
    'pseudo_mix_ratio': float(PSEUDO_MIX_RATIO),
}
with open(run_dir / 'run_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('Run dir        :', run_dir)
print('Run submission :', run_submission_path)
print('Platform output:', platform_submission_path)
if MODE == 'local':
    print('Local copy     :', local_submission_path)

print('\nSubmission class distribution:')
print(submission['disease'].value_counts())
submission.head()
